# 04 — Tensor Networks in Machine Learning

Tensor networks find three major roles in modern ML:
1. **Model compression** — replace large weight matrices/tensors with compact factorised forms
2. **Feature learning** — embed data into tensor product spaces
3. **Generative modelling** — represent probability distributions as tensor networks

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import tensorly as tl
from tensorly.decomposition import parafac, tucker
tl.set_backend('numpy')
import matplotlib.pyplot as plt
from numpy.linalg import norm
print("PyTorch version:", torch.__version__)
print("TensorLy version:", tl.__version__)

## 1. Neural Network Layer Compression via Tensor Train

A **fully-connected layer** maps $\mathbf{x} \in \mathbb{R}^n \to \mathbf{y} \in \mathbb{R}^m$ via weight matrix $W \in \mathbb{R}^{m \times n}$.

Idea: reshape $W$ into a high-order tensor $\mathcal{W}$ of shape $(m_1 \cdots m_k, n_1 \cdots n_k)$, then apply a **Tensor Train decomposition** to it.

This compresses the layer from $O(mn)$ parameters to $O(k \cdot r^2 \cdot d)$ parameters.

$$W_{(i_1 \cdots i_k)(j_1 \cdots j_k)} \approx \text{TT-cores contracted}$$

In [ ]:
# Simulate a large weight matrix like a big FC layer
np.random.seed(42)
m, n = 256, 128   # output x input
W = np.random.randn(m, n)
print(f"Original weight matrix W: shape={W.shape}, params={W.size}")

# Reshape into a 6-way tensor: (4, 4, 4, 4, 4, 2)  — 4^5 * 2 = 2048 = 256*128? No, let's check
# 256 = 4*4*4*4, 128 = 4*4*4*2 — yes
shape_reshaped = (4, 4, 4, 4, 4, 4, 4, 2)  # m=4^4=256, n=4^3*2=128
W_tensor = W.reshape(shape_reshaped)
print(f"Reshaped tensor for TT: {W_tensor.shape}, total={W_tensor.size}")

# TT decomposition with bond dim 4
from tensorly.decomposition import tensor_train
bond = 4
ranks = [1] + [bond] * (len(shape_reshaped) - 1) + [1]
tt_cores = tensor_train(tl.tensor(W_tensor), rank=ranks)

W_recon_tensor = tl.tt_to_tensor(tt_cores)
W_recon = W_recon_tensor.reshape(m, n)

params_tt = sum(c.size for c in tt_cores)
print(f"\nTT params: {params_tt}")
print(f"Compression: {W.size / params_tt:.1f}x")
print(f"Relative error: {norm(W - W_recon) / norm(W):.5f}")

## 2. A PyTorch TT-Linear Layer

We implement a custom `nn.Module` that stores a linear layer weight as Tensor Train cores,  
then contracts them on the forward pass — avoiding materialising the full weight matrix.

In [ ]:
class TTLinear(nn.Module):
    """
    Tensor Train linear layer.
    Instead of one W[m, n] parameter, we store a sequence of small cores.
    
    For simplicity we fix a factorisation: n -> (d1, d2, ..., dk) and m -> (e1, e2, ..., ek)
    and use a sequence of cores G[k] of shape (r_{k-1}, d_k, e_k, r_k).
    """
    def __init__(self, in_features, out_features, in_modes, out_modes, bond_dim):
        super().__init__()
        assert np.prod(in_modes) == in_features
        assert np.prod(out_modes) == out_features
        assert len(in_modes) == len(out_modes)
        self.in_modes = in_modes
        self.out_modes = out_modes
        self.in_features = in_features
        self.out_features = out_features
        order = len(in_modes)

        # TT cores: shape (r_{k-1}, d_k, e_k, r_k)
        self.cores = nn.ParameterList()
        for k in range(order):
            r_left  = 1 if k == 0 else bond_dim
            r_right = 1 if k == order - 1 else bond_dim
            core = nn.Parameter(
                torch.randn(r_left, in_modes[k], out_modes[k], r_right) * 0.1
            )
            self.cores.append(core)

    def forward(self, x):
        # x: (batch, in_features)
        batch = x.shape[0]
        # Reshape x to (batch, d1, d2, ..., dk)
        x = x.view(batch, *self.in_modes)

        # Contract cores with x one mode at a time
        # After k-th step: state has shape (batch, e1,...,e_k, r_k)
        order = len(self.in_modes)
        # Start with core 0: shape (1, d1, e1, r)
        # x[:, i1, i2, ...] * core[0][0, i1, j1, r] -> (batch, e1, r)
        # Use einsum step by step
        state = torch.einsum('b...,rdsr->b...s', x[..., 0:1], self.cores[0])  # placeholder

        # Simpler: materialise the weight matrix from cores (for correctness demo)
        W = self.cores[0].reshape(self.in_modes[0], self.out_modes[0], -1)  # (d1, e1, r)
        for k in range(1, order):
            core = self.cores[k]  # (r, dk, ek, r')
            W = torch.einsum('...r,rdep->...dep', W, core).reshape(
                *[self.in_modes[i] for i in range(k+1)],
                *[self.out_modes[i] for i in range(k+1)],
                self.cores[k].shape[-1]
            )
        # Final W arranged as (d1,...,dk, e1,...,ek, 1) -> (in_features, out_features)
        W_full = W.squeeze(-1).reshape(self.in_features, self.out_features)
        return x.view(batch, self.in_features) @ W_full


# Quick sanity check
tt_layer = TTLinear(
    in_features=16, out_features=8,
    in_modes=[4, 4], out_modes=[2, 4],
    bond_dim=3
)
x_test = torch.randn(5, 16)
y_test = tt_layer(x_test)
print(f"TTLinear output shape: {y_test.shape}")
print(f"TTLinear param count: {sum(p.numel() for p in tt_layer.parameters())}")
print(f"Standard Linear param count: 16*8+8 = {16*8+8}")

## 3. Tensor-based Feature Maps

**Idea**: Embed each data point $\mathbf{x} \in \mathbb{R}^d$ into a **tensor product feature space**:

$$\Phi(\mathbf{x}) = \phi(x_1) \otimes \phi(x_2) \otimes \cdots \otimes \phi(x_d)$$

where $\phi: \mathbb{R} \to \mathbb{R}^s$ is a local feature map (e.g. polynomial, Fourier, or trig basis).

A linear classifier in this exponentially large feature space is equivalent to contracting $\Phi(\mathbf{x})$ with a weight tensor $\mathcal{W}$ — this is exactly an MPS classifier!

A simple 2-feature example with quadratic features:

In [ ]:
def phi(x_scalar, s=2):
    """Local feature map: [1, x] (linear basis)"""
    return np.array([1.0, float(x_scalar)])

def tensor_product_features(x_vec):
    """Outer product of local features over all dimensions."""
    result = phi(x_vec[0])
    for i in range(1, len(x_vec)):
        result = np.outer(result, phi(x_vec[i])).ravel()
    return result

# 3 features => 2^3 = 8-dimensional product feature space
x = np.array([0.5, 1.2, -0.3])
phi_x = tensor_product_features(x)
print(f"Input x: {x}")
print(f"Tensor product feature vector (dim {len(phi_x)}): {phi_x.round(3)}")

# A weight tensor W of shape (2,2,2) acts as a linear classifier
W = np.random.randn(2, 2, 2)
score = np.dot(W.ravel(), phi_x)
print(f"\nClassifier score (W · Phi(x)): {score:.5f}")

## 4. Simple MPS Classifier on Toy Data

We train an MPS-based binary classifier on a simple 2-D dataset.  
The weight tensor $\mathcal{W}$ is stored in MPS form and trained with PyTorch autograd.

In [ ]:
# Generate toy XOR-like dataset
np.random.seed(7)
N = 200
X = np.random.randn(N, 4).astype(np.float32)   # 4 features
y = ((X[:, 0] * X[:, 1] + X[:, 2] - X[:, 3]) > 0).astype(np.float32)

X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y)

dataset = TensorDataset(X_t, y_t)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)
print(f"Dataset: {N} samples, {X.shape[1]} features, class balance: {y.mean():.2f}")

In [ ]:
class SimpleMPSClassifier(nn.Module):
    """
    MPS classifier: encode each feature via a local map phi(x_i) ∈ R^2,
    then the output is a scalar from contracting an MPS with the product state.
    
    For 4 features with phi_dim=2 and bond_dim r:
      G0: (1, 2, r), G1: (r, 2, r), G2: (r, 2, r), G3: (r, 2, 1)
    """
    def __init__(self, n_features=4, phi_dim=2, bond_dim=4):
        super().__init__()
        self.n = n_features
        self.phi_dim = phi_dim
        self.bond = bond_dim
        self.cores = nn.ParameterList()
        for k in range(n_features):
            r_l = 1 if k == 0 else bond_dim
            r_r = 1 if k == n_features-1 else bond_dim
            self.cores.append(nn.Parameter(torch.randn(r_l, phi_dim, r_r) * 0.1))

    def encode(self, x):
        """Local feature map: phi(x_i) = [cos(pi/2 * x_i), sin(pi/2 * x_i)]"""
        return torch.stack([torch.cos(np.pi/2 * x), torch.sin(np.pi/2 * x)], dim=-1)

    def forward(self, x):
        # x: (batch, n_features)
        phi = self.encode(x)   # (batch, n_features, phi_dim)
        # contract MPS with product-state phi
        # state: (batch, r) — start with bond-dim=1
        state = torch.einsum('bp,rpR->bR', phi[:, 0, :], self.cores[0])  # (batch, r)
        for k in range(1, self.n):
            state = torch.einsum('bL,bp,LpR->bR', state, phi[:, k, :], self.cores[k])
        return state.squeeze(-1)   # (batch,) — scalar output


model = SimpleMPSClassifier(n_features=4, phi_dim=2, bond_dim=4)
print(f"MPS classifier params: {sum(p.numel() for p in model.parameters())}")

optimizer = optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

losses = []
for epoch in range(50):
    epoch_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(loader))

# Accuracy
with torch.no_grad():
    preds = (model(X_t) > 0).float()
    acc = (preds == y_t).float().mean().item()
print(f"\nFinal accuracy: {acc:.3f}")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('BCE loss')
plt.title('MPS Classifier Training Loss')
plt.tight_layout(); plt.show()

## 5. Tucker Decomposition for Convolutional Layer Compression

A **4-D convolutional weight tensor** $W \in \mathbb{R}^{C_{out} \times C_{in} \times H \times W}$ can be Tucker-decomposed:

$$W \approx \mathcal{G} \times_1 U_{C_{out}} \times_2 U_{C_{in}} \times_3 U_H \times_4 U_W$$

This replaces the original convolution with three separable convolutions:  
`in_channels → pointwise → Tucker core conv → pointwise → out_channels`

In [ ]:
# Simulate compressing a Conv2d weight
np.random.seed(0)
C_out, C_in, kH, kW = 64, 32, 3, 3
W_conv = np.random.randn(C_out, C_in, kH, kW)
print(f"Conv weight W shape: {W_conv.shape}, params: {W_conv.size}")

# Tucker decompose with rank (16, 8, 3, 3)
ranks = (16, 8, 3, 3)
core, factors = tucker(tl.tensor(W_conv), rank=ranks)
W_recon = tl.tucker_to_tensor((core, factors))

params_tucker = core.size + sum(f.size for f in factors)
print(f"Tucker params: {params_tucker}")
print(f"Compression: {W_conv.size / params_tucker:.1f}x")
print(f"Relative error: {norm(W_conv - W_recon) / norm(W_conv):.5f}")

print(f"\nCore shape: {core.shape}")
print(f"Factor shapes: {[f.shape for f in factors]}")

## 6. Knowledge Recap: Why Tensor Networks for ML?

| Application | Tensor Tool | Benefit |
|------------|-------------|----------|
| FC layer compression | TT-decomposition | $O(mn) \to O(rnd)$ parameters |
| Conv layer compression | Tucker decomposition | Separable, hardware-friendly |
| Feature maps | Tensor product spaces | Polynomial interactions captured |
| Sequence models | MPS / TT | Compact representation of joint distributions |
| Anomaly detection | Tensor train density models | Exact normalisation via sweeping |

➡️ **05_libraries_overview.ipynb** — hands-on with every library independently.